# DeePoo EfficeintDet-Lite0 Training & Evaluation

This notebook demonstrates how to train and evaluate a EffcientDet model for poo detection using a COCO-formatted dataset.

**System RAM**: 51.0 GB

**GPU**: T4

**GPU RAM**: 15.0 GB

**Dataset**: `poo_base_640x640` in COCO format (https://cocodataset.org/#format-data)

**Model**: EfficientDet-Lite0 (https://github.com/google/automl/tree/master/efficientdet)

**Image size**: 320x320

**Data augmentation policy**: None

**Train epochs**: 50

**Train precision**: float32

**Export precision**: float32

**Virtual environment**: Konda

**Framework**: Tensorflow

**Environment**: Google Colab

## 0. Install Required Libraries

### Install Konda

In [ ]:
!pip install konda

In [ ]:
import konda
konda.install()

In [ ]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

In [ ]:
!konda run "conda install mamba -n base -c conda-forge -y"

### Create enviroment to run AutoML

In [ ]:
!konda create -n automl_env python=3.10 -y

In [ ]:
!konda activate automl_env

### Install required libraries

In [ ]:
!pip install tensorflow

In [ ]:
!konda run "mamba install conda-forge::tensorflow=2.8 conda-forge::pycocotools pyyaml -y"

In [ ]:
!konda run "pip install tensorflow-addons"

### Clone AutoML

In [ ]:
import sys

# Clone the AutoML repository (contains EfficientDet implementation)
print("\n📥 Cloning Google AutoML repository...")
!git clone --depth 1 https://github.com/google/automl.git /content/automl 2>/dev/null || echo "Repository already exists"

## 1. Configuration


Select your EfficientDet model variant

Available variants:

| Variant             | Parameters |
|---------------------|:----------:|
| efficientdet-lite0  |    3.2M    |
| efficientdet-lite1  |    4.2M    |
| efficientdet-lite2  |    5.3M    |
| efficientdet-lite3  |    8.4M    |
| efficientdet-lite3x |    9.3M    |
| efficientdet-lite4  |   15.1M    |

In [ ]:
from pathlib import Path
import os

# Model Variant
MODEL_VARIANT = 'efficientdet-lite0'  # Change this to your preferred variant

# Training Configuration

AUGMENTATION_POLICY = None
TRAIN_PRECISION = "FP32"
BATCH_SIZE = 16
NUM_EPOCHS = 50
IMAGE_SIZE = 320
NUM_CLASSES = 1  # Number of object classes (excluding background)
LEARNING_RATE = 0.08

# Dataset Configuration
DATASET_EXTRACT_PATH = Path('/content/dataset')
DATASET_ZIP_PATH = '/content/poo_base_640x640.zip'
DATASET_NAME = os.path.splitext(os.path.basename(DATASET_ZIP_PATH))[0]
DATASET_ROOT = DATASET_EXTRACT_PATH / DATASET_NAME

# Training Paths
CHECKPOINT_DIR = Path('/content/checkpoints')

# Export Path
EXPORT_PRECISION = "FP32"
OUTPUT_DIR = Path('/content/output')


# Model Variant Specifications
MODEL_SPECS = {
    'efficientdet-lite0': {'recommended_resolution': 320, 'recommended_batch': 16},
    'efficientdet-lite1': {'recommended_resolution': 384, 'recommended_batch': 12},
    'efficientdet-lite2': {'recommended_resolution': 448, 'recommended_batch': 8},
    'efficientdet-lite3': {'recommended_resolution': 512, 'recommended_batch': 6},
    'efficientdet-lite3x': {'recommended_resolution': 512, 'recommended_batch': 4},
    'efficientdet-lite4': {'recommended_resolution': 512, 'recommended_batch': 4},

}

# Check image size based on model variant
if MODEL_VARIANT in MODEL_SPECS:
    print(f"✓ Selected Model: {MODEL_VARIANT}")
    if IMAGE_SIZE != MODEL_SPECS[MODEL_VARIANT]['recommended_resolution']:
        print(f"⚠ Warning: Image Resolution: {IMAGE_SIZE}x{IMAGE_SIZE} may not be optimal for {MODEL_VARIANT}")
    else:
        print(f"✓ Image Resolution: {IMAGE_SIZE}x{IMAGE_SIZE}")
    if BATCH_SIZE != MODEL_SPECS[MODEL_VARIANT]['recommended_batch']:
        print(f"⚠ Warning: Batch Size: {BATCH_SIZE} may not be optimal for {MODEL_VARIANT}")
    else:
        print(f"✓ Batch Size: {BATCH_SIZE}")

    if BATCH_SIZE > MODEL_SPECS[MODEL_VARIANT]['recommended_batch']:
        print(f"⚠ Warning: Batch size may be too large for GPU memory")
else:
    print(f"❌ Error: Unknown model variant '{MODEL_VARIANT}'")
    print(f"Available variants: {list(MODEL_SPECS.keys())}")

##

## 2. Dataset Setup and Extraction Dataset

In [ ]:
from re import I
import zipfile
from pathlib import Path

# Create necessary directories
os.makedirs(DATASET_EXTRACT_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📂 Extracting dataset...")
print(f"   Source: {DATASET_ZIP_PATH}")
print(f"   Destination: {DATASET_EXTRACT_PATH}")

# Extract the dataset
with zipfile.ZipFile(DATASET_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(DATASET_EXTRACT_PATH)

print("✅ Dataset extracted successfully!")

print(f"\n📁 Dataset structure:")
for item in sorted(DATASET_ROOT.iterdir()):
    if item.is_dir():
        num_files = len(list(item.rglob('*')))
        print(f"   📂 {item.name}/ ({num_files} items)")
    else:
        print(f"   📄 {item.name}")

# COCO annotation files
ANNOTATIONS_DIR = DATASET_ROOT / 'annotations'
TRAIN_ANNOTATIONS = ANNOTATIONS_DIR / 'instances_train.json'
VAL_ANNOTATIONS = ANNOTATIONS_DIR / 'instances_val.json'
TEST_ANNOTATIONS = ANNOTATIONS_DIR / 'instances_test.json'

# Image files
IMAGES_DIR = DATASET_ROOT / 'images'
TRAIN_IMAGES_DIR = IMAGES_DIR / 'train'
VAL_IMAGES_DIR = IMAGES_DIR / 'val'
TEST_IMAGES_DIR = IMAGES_DIR / 'test'

print(f"\n📸 Image directories:")
print(f"   Train: {TRAIN_IMAGES_DIR}")
print(f"   Val: {VAL_IMAGES_DIR}")
print(f"   Test: {TEST_IMAGES_DIR}")

## 3. Import Required Libraries

In [ ]:
# Standard libraries
import sys
import random
import json
import numpy as np
import pandas as pd

# Image processing
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches

# TensorFlow and Keras
import tensorflow as tf

# COCO tools
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

## 4. Device Setup and Configuration

### Colab runtime

In [ ]:
%%writefile device_setup.py

import tensorflow as tf

# Check GPU availability and configure
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        # Enable memory growth to prevent TensorFlow from allocating all GPU memory
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        # Set the GPU as the default device
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f"✅ GPU Configuration:")
        print(f"   Physical GPUs: {len(gpus)}")
        print(f"   Logical GPUs: {len(logical_gpus)}")
        print(f"   GPU Name: {gpus[0].name}")

        # Get GPU details
        gpu_details = tf.config.experimental.get_device_details(gpus[0])
        if 'device_name' in gpu_details:
            print(f"   GPU Model: {gpu_details['device_name']}")

        DEVICE = '/GPU:0'

    except RuntimeError as e:
        print(f"⚠️  GPU setup error: {e}")
        DEVICE = '/CPU:0'
else:
    print("⚠️  No GPU detected, using CPU")
    DEVICE = '/CPU:0'

print(f"\n🖥️  Training Device: {DEVICE}")

# Configure mixed precision for faster training (if GPU available)
if gpus:
    try:
        policy = tf.keras.mixed_precision.Policy('mixed_float16')
        tf.keras.mixed_precision.set_global_policy(policy)
        print(f"✅ Mixed precision enabled: {policy.name}")
        print(f"   Compute dtype: {policy.compute_dtype}")
        print(f"   Variable dtype: {policy.variable_dtype}")
    except:
        print("⚠️  Mixed precision not available, using float32")

# Display TensorFlow configuration
print(f"\n📊 TensorFlow Configuration:")
print(f"   Version: {tf.__version__}")
print(f"   Eager Execution: {tf.executing_eagerly()}")
print(f"   Built with CUDA: {tf.test.is_built_with_cuda()}")

In [ ]:
!python device_setup.py

### AutoML environment

In [ ]:
!konda run "python device_setup.py"

## 5. Explore Dataset

### Load dataset

In [ ]:
print("📊 Loading COCO annotations...")

# Load training annotations
coco_train = COCO(TRAIN_ANNOTATIONS)
print(f"✅ Training annotations loaded")

# Load validation annotations
coco_val = COCO(VAL_ANNOTATIONS)
print(f"✅ Validation annotations loaded")

# Load test annotations
coco_test = COCO(TEST_ANNOTATIONS)
print(f"✅ Test annotations loaded")

# Get dataset statistics
train_img_ids = coco_train.getImgIds()
val_img_ids = coco_val.getImgIds()
test_img_ids = coco_test.getImgIds()

train_cat_ids = coco_train.getCatIds()
val_cat_ids = coco_val.getCatIds()
test_cat_ids = coco_test.getCatIds()

train_categories = coco_train.loadCats(train_cat_ids)
val_categories = coco_val.loadCats(val_cat_ids)
test_categories = coco_test.loadCats(test_cat_ids)

# Count annotations
train_ann_ids = coco_train.getAnnIds()
val_ann_ids = coco_val.getAnnIds()
test_ann_ids = coco_test.getAnnIds()

print(f"\n📈 Dataset Statistics:")
print(f"   {'='*50}")
print(f"   Training Set:")
print(f"      Images: {len(train_img_ids)}")
print(f"      Annotations: {len(train_ann_ids)}")
print(f"      Categories: {len(train_categories)}")
print(f"   {'='*50}")
print(f"   Validation Set:")
print(f"      Images: {len(val_img_ids)}")
print(f"      Annotations: {len(val_ann_ids)}")
print(f"      Categories: {len(val_categories)}")
print(f"   {'='*50}")
print(f"   Test Set:")
print(f"      Images: {len(test_img_ids)}")
print(f"      Annotations: {len(test_ann_ids)}")
print(f"      Categories: {len(test_categories)}")

# Display category information
print(f"\n🏷️  Categories:")
for cat in train_categories:
    # Count annotations for this category
    ann_ids = coco_train.getAnnIds(catIds=[cat['id']])
    print(f"   ID {cat['id']}: {cat['name']} ({len(ann_ids)} annotations)")

# Calculate average annotations per image
avg_train_ann = len(train_ann_ids) / len(train_img_ids) if train_img_ids else 0
avg_val_ann = len(val_ann_ids) / len(val_img_ids) if val_img_ids else 0
avg_test_ann = len(test_ann_ids) / len(test_img_ids) if val_img_ids else 0

print(f"\n📊 Annotation Density:")
print(f"   Training: {avg_train_ann:.2f} annotations per image")
print(f"   Validation: {avg_val_ann:.2f} annotations per image")
print(f"   Test: {avg_test_ann:.2f} annotaions per image")

# Sample image information
if train_img_ids:
    sample_img = coco_train.loadImgs(train_img_ids[0])[0]
    print(f"\n🖼️  Sample Image Info:")
    print(f"   File: {sample_img['file_name']}")
    print(f"   Size: {sample_img['width']}x{sample_img['height']}")
    print(f"   ID: {sample_img['id']}")

# Store for later use
COCO_TRAIN = coco_train
COCO_VAL = coco_val
COCO_TEST = coco_test
TRAIN_IMG_IDS = train_img_ids
VAL_IMG_IDS = val_img_ids
TEST_IMG_IDS = test_img_ids
CATEGORY_NAMES = [cat['name'] for cat in train_categories]
CATEGORY_IDS = [cat['id'] for cat in train_categories]

print(f"\n✅ Dataset exploration complete!")

## Visualize samples with annotations

In [ ]:
def visualize_coco_samples(coco, img_ids, images_dir, num_samples=4, cols=2):
    """Visualize sample images with bounding box annotations"""

    num_samples = min(num_samples, len(img_ids))
    rows = (num_samples + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(15, 7*rows))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1 or cols == 1:
        axes = axes.reshape(rows, cols)

    # Randomly sample images
    sample_ids = random.sample(img_ids, num_samples)

    for idx, img_id in enumerate(sample_ids):
        row = idx // cols
        col = idx % cols
        ax = axes[row, col]

        # Load image info
        img_info = coco.loadImgs(img_id)[0]
        img_path = os.path.join(images_dir, img_info['file_name'])

        # Load and display image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)

        # Get annotations for this image
        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)

        # Draw bounding boxes
        for ann in anns:
            bbox = ann['bbox']  # [x, y, width, height]
            x, y, w, h = bbox

            # Get category info
            cat = coco.loadCats(ann['category_id'])[0]

            # Draw rectangle
            rect = patches.Rectangle(
                (x, y), w, h,
                linewidth=2,
                edgecolor='fuchsia',
                facecolor='none'
            )
            ax.add_patch(rect)

            # Add label
            label = f"{cat['name']}"
            ax.text(
                x, y - 5,
                label,
                color='white',
                fontsize=10,
                bbox=dict(facecolor='fuchsia', alpha=0.7, edgecolor='none', pad=2)
            )

        ax.set_title(f"{img_info['file_name']}\n{len(anns)} annotations", fontsize=10)
        ax.axis('off')

    # Hide empty subplots
    for idx in range(num_samples, rows * cols):
        row = idx // cols
        col = idx % cols
        axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()

# Visualize training samples
print("🖼️  Training Set Samples:")
visualize_coco_samples(COCO_TRAIN, TRAIN_IMG_IDS, TRAIN_IMAGES_DIR, num_samples=3, cols=3)

# Visualize validation samples
print("\n🖼️  Validation Set Samples:")
visualize_coco_samples(COCO_VAL, VAL_IMG_IDS, VAL_IMAGES_DIR, num_samples=3, cols=3)

# Visualize test samples
print("\n🖼️  Test Set Samples:")
visualize_coco_samples(COCO_TEST, TEST_IMG_IDS, TEST_IMAGES_DIR, num_samples=3, cols=3)

## 6. Data Preparation

In [ ]:
print("🔄 Converting COCO dataset to TFRecord format...")
print("   This is required for EfficientDet training\n")

# Create TFRecord output directory
TFRECORD_DIR = '/content/tfrecords'
os.makedirs(TFRECORD_DIR, exist_ok=True)

def create_tf_example(image_path, image_id, annotations, image_info):
    """Create a TF Example from COCO annotation"""

    # Read image
    with tf.io.gfile.GFile(image_path, 'rb') as fid:
        encoded_image = fid.read()

    # Get image dimensions
    height = image_info['height']
    width = image_info['width']
    filename = image_info['file_name'].encode('utf8')
    image_format = b'jpg'

    # Prepare bounding boxes and labels
    xmins = []
    xmaxs = []
    ymins = []
    ymaxs = []
    classes_text = []
    classes = []
    areas = []

    for ann in annotations:
        bbox = ann['bbox']  # [x, y, width, height] in COCO format
        x, y, w, h = bbox

        # Normalize coordinates to [0, 1]
        xmin = max(0.0, x / width)
        ymin = max(0.0, y / height)
        xmax = min(1.0, (x + w) / width)
        ymax = min(1.0, (y + h) / height)

        # Skip invalid boxes
        if xmax <= xmin or ymax <= ymin:
            continue

        xmins.append(xmin)
        xmaxs.append(xmax)
        ymins.append(ymin)
        ymaxs.append(ymax)

        # Category (subtract 1 because EfficientDet expects 0-indexed classes)
        class_id = ann['category_id']
        classes.append(class_id)
        classes_text.append(CATEGORY_NAMES[0].encode('utf8'))  # 'poo'

        # Area
        areas.append(ann.get('area', w * h))

    # Create TF Example
    tf_example = tf.train.Example(features=tf.train.Features(feature={
        'image/height': tf.train.Feature(int64_list=tf.train.Int64List(value=[height])),
        'image/width': tf.train.Feature(int64_list=tf.train.Int64List(value=[width])),
        'image/filename': tf.train.Feature(bytes_list=tf.train.BytesList(value=[filename])),
        'image/source_id': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(image_id).encode('utf8')])),
        'image/encoded': tf.train.Feature(bytes_list=tf.train.BytesList(value=[encoded_image])),
        'image/format': tf.train.Feature(bytes_list=tf.train.BytesList(value=[image_format])),
        'image/object/bbox/xmin': tf.train.Feature(float_list=tf.train.FloatList(value=xmins)),
        'image/object/bbox/xmax': tf.train.Feature(float_list=tf.train.FloatList(value=xmaxs)),
        'image/object/bbox/ymin': tf.train.Feature(float_list=tf.train.FloatList(value=ymins)),
        'image/object/bbox/ymax': tf.train.Feature(float_list=tf.train.FloatList(value=ymaxs)),
        'image/object/class/text': tf.train.Feature(bytes_list=tf.train.BytesList(value=classes_text)),
        'image/object/class/label': tf.train.Feature(int64_list=tf.train.Int64List(value=classes)),
        'image/object/area': tf.train.Feature(float_list=tf.train.FloatList(value=areas)),
    }))

    return tf_example

def convert_coco_to_tfrecord(coco, img_ids, images_dir, output_path, split_name):
    """Convert COCO dataset to TFRecord"""

    writer = tf.io.TFRecordWriter(output_path)

    print(f"   Converting {split_name} set ({len(img_ids)} images)...")

    skipped = 0
    for idx, img_id in enumerate(img_ids):
        if (idx + 1) % 100 == 0:
            print(f"      Progress: {idx + 1}/{len(img_ids)}")

        # Load image info
        img_info = coco.loadImgs(img_id)[0]
        image_path = os.path.join(images_dir, img_info['file_name'])

        # Check if image exists
        if not os.path.exists(image_path):
            print(f"      Warning: Image not found: {image_path}")
            skipped += 1
            continue

        # Get annotations
        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)

        # Skip images without annotations
        if len(anns) == 0:
            skipped += 1
            continue

        # Create TF Example
        try:
            tf_example = create_tf_example(image_path, img_id, anns, img_info)
            writer.write(tf_example.SerializeToString())
        except Exception as e:
            print(f"      Error processing image {img_info['file_name']}: {e}")
            skipped += 1

    writer.close()
    print(f"   ✅ {split_name} set complete! ({len(img_ids) - skipped} images written, {skipped} skipped)")

# Convert training set
train_tfrecord_path = os.path.join(TFRECORD_DIR, 'train.tfrecord')
convert_coco_to_tfrecord(COCO_TRAIN, TRAIN_IMG_IDS, TRAIN_IMAGES_DIR, train_tfrecord_path, 'Training')

# Convert validation set
val_tfrecord_path = os.path.join(TFRECORD_DIR, 'val.tfrecord')
convert_coco_to_tfrecord(COCO_VAL, VAL_IMG_IDS, VAL_IMAGES_DIR, val_tfrecord_path, 'Validation')

# Convert test set
test_tfrecord_path = os.path.join(TFRECORD_DIR, 'test.tfrecord')
convert_coco_to_tfrecord(COCO_TEST, TEST_IMG_IDS, TEST_IMAGES_DIR, test_tfrecord_path, 'Test')

print(f"\n✅ TFRecord conversion complete!")
print(f"   Training TFRecord: {train_tfrecord_path}")
print(f"   Validation TFRecord: {val_tfrecord_path}")
print(f"   Test TFRecord: {test_tfrecord_path}")

# Store paths for training
TRAIN_TFRECORD = train_tfrecord_path
VAL_TFRECORD = val_tfrecord_path
TEST_TFRECORD = test_tfrecord_path

## 7. Model Setup

### Download pretrained weights

In [ ]:
print(f"\n📥 Downloading pretrained EfficientDet-Lite0 weights...")

PRETRAINED_DIR = CHECKPOINT_DIR / 'pretrained'
PRETRAINED_CKPT = PRETRAINED_DIR / MODEL_VARIANT
WEIGHTS_URL = f'https://storage.googleapis.com/cloud-tpu-checkpoints/efficientdet/coco/{MODEL_VARIANT}.tgz'

# Download and extract weights
import urllib.request
import tarfile

weights_file = f'/content/{MODEL_VARIANT}.tgz'

try:
    if not os.path.exists(PRETRAINED_CKPT):
        print(f"   Downloading from: {WEIGHTS_URL}")
        urllib.request.urlretrieve(WEIGHTS_URL, weights_file)

        print(f"   Extracting weights...")
        with tarfile.open(weights_file, 'r:gz') as tar:
            tar.extractall(PRETRAINED_DIR)

        print(f"✅ Pretrained weights downloaded and extracted")
    else:
        print(f"✅ Pretrained weights already exist")

except Exception as e:
    print(f"⚠️  Could not download pretrained weights: {e}")

## 8. Training Configuration

In [ ]:
HPARAMS_DICT = dict (
    name = MODEL_VARIANT,
    num_classes = NUM_CLASSES,
    image_size = IMAGE_SIZE,
    autoaugment_policy = AUGMENTATION_POLICY,
    learning_rate = LEARNING_RATE,
    label_map = {0: 'poo'},
    mixed_precision = TRAIN_PRECISION != 'FP32',
)

HPARAMS_STR = ', '.join([f'{k}={v}' for k, v in HPARAMS_DICT.items()])
HPARAMS_STR

## 9. Training

### Load and launch TensorBoard extension

In [ ]:
%load_ext tensorboard

%tensorboard --logdir {CHECKPOINT_DIR}

In [ ]:
!konda run "python automl/efficientdet/main.py \
    --mode=train_and_eval \
    --model_name={MODEL_VARIANT} \
    --train_file_pattern={TRAIN_TFRECORD} \
    --val_file_pattern={VAL_TFRECORD} \
    --val_json_file={str(VAL_ANNOTATIONS)} \
    --model_dir={str(CHECKPOINT_DIR)} \
    --num_epochs={NUM_EPOCHS}  \
    --ckpt={PRETRAINED_CKPT}  \
    --train_batch_size={BATCH_SIZE} \
    --eval_batch_size={BATCH_SIZE} \
    --num_examples_per_epoch={len(TRAIN_IMG_IDS)} \
    --hparams={HPARAMS_STR}"

## 10. Training Results Analysis

### Load TensorBoard event files

In [ ]:
import glob

print("\n📊 Loading training metrics from TensorBoard logs...")

try:
    from tensorflow.python.summary.summary_iterator import summary_iterator

    # Find event files
    event_files = glob.glob(str(CHECKPOINT_DIR / "eval" / 'events.out.tfevents.*'))

    if event_files:
        # Collect all events first to handle different metric frequencies
        all_events = []

        for event_file in event_files:
            for event in summary_iterator(event_file):
                event_metrics = {'step': event.step}
                for value in event.summary.value:
                    if value.tag in ['loss', 'cls_loss', 'box_loss', 'AP', 'AP50', 'AP75', 'ARmax100']:
                        event_metrics[value.tag] = value.simple_value
                if len(event_metrics) > 1:  # Only keep events with actual metrics
                    all_events.append(event_metrics)

        if all_events:
            # Convert to DataFrame to handle missing values properly
            df_metrics = pd.DataFrame(all_events)

            # Sort by step and forward fill missing values for smoother plots
            df_metrics = df_metrics.sort_values('step').reset_index(drop=True)

            # Optional: Forward fill missing values for continuous metrics
            for col in ['loss', 'cls_loss', 'box_loss']:
                if col in df_metrics.columns:
                    df_metrics[col] = df_metrics[col].fillna(method='ffill')

            if not df_metrics.empty:
                # Plot Training Losses
                fig, axes = plt.subplots(2, 2, figsize=(16, 12))

                # Total Loss
                if 'loss' in df_metrics.columns and not df_metrics['loss'].isna().all():
                    axes[0, 0].plot(df_metrics['step'], df_metrics['loss'], linewidth=2, color='#2E86AB')
                    axes[0, 0].set_title('Total Loss Over Training', fontsize=14, fontweight='bold')
                    axes[0, 0].set_xlabel('Training Steps')
                    axes[0, 0].set_ylabel('Loss')
                    axes[0, 0].grid(True, alpha=0.3)

                # Classification Loss
                if 'cls_loss' in df_metrics.columns and not df_metrics['cls_loss'].isna().all():
                    axes[0, 1].plot(df_metrics['step'], df_metrics['cls_loss'], linewidth=2, color='#A23B72')
                    axes[0, 1].set_title('Classification Loss', fontsize=14, fontweight='bold')
                    axes[0, 1].set_xlabel('Training Steps')
                    axes[0, 1].set_ylabel('Classification Loss')
                    axes[0, 1].grid(True, alpha=0.3)

                # Box Regression Loss
                if 'box_loss' in df_metrics.columns and not df_metrics['box_loss'].isna().all():
                    axes[1, 0].plot(df_metrics['step'], df_metrics['box_loss'], linewidth=2, color='#F18F01')
                    axes[1, 0].set_title('Box Regression Loss', fontsize=14, fontweight='bold')
                    axes[1, 0].set_xlabel('Training Steps')
                    axes[1, 0].set_ylabel('Box Loss')
                    axes[1, 0].grid(True, alpha=0.3)

                # Combined Loss View
                if all(col in df_metrics.columns for col in ['cls_loss', 'box_loss']):
                    axes[1, 1].plot(df_metrics['step'], df_metrics['cls_loss'],
                                   linewidth=2, label='Classification Loss', color='#A23B72')
                    axes[1, 1].plot(df_metrics['step'], df_metrics['box_loss'],
                                   linewidth=2, label='Box Loss', color='#F18F01')
                    axes[1, 1].set_title('Loss Components Comparison', fontsize=14, fontweight='bold')
                    axes[1, 1].set_xlabel('Training Steps')
                    axes[1, 1].set_ylabel('Loss')
                    axes[1, 1].legend()
                    axes[1, 1].grid(True, alpha=0.3)

                plt.tight_layout()
                plt.savefig(OUTPUT_DIR / 'training_losses.png', dpi=300, bbox_inches='tight')
                plt.show()

                # Plot mAP Metrics
                fig, axes = plt.subplots(2, 2, figsize=(16, 12))

                # AP (mAP@0.5:0.95)
                if 'AP' in df_metrics.columns and not df_metrics['AP'].isna().all():
                    axes[0, 0].plot(df_metrics['step'], df_metrics['AP'], linewidth=2,
                                   color='#06A77D', marker='o', markersize=4)
                    axes[0, 0].set_title('mAP@[0.5:0.95] Over Training', fontsize=14, fontweight='bold')
                    axes[0, 0].set_xlabel('Training Steps')
                    axes[0, 0].set_ylabel('mAP')
                    axes[0, 0].set_ylim([0, 1])
                    axes[0, 0].grid(True, alpha=0.3)

                # AP50 (mAP@0.5)
                if 'AP50' in df_metrics.columns and not df_metrics['AP50'].isna().all():
                    axes[0, 1].plot(df_metrics['step'], df_metrics['AP50'], linewidth=2,
                                   color='#118AB2', marker='o', markersize=4)
                    axes[0, 1].set_title('mAP@0.5 Over Training', fontsize=14, fontweight='bold')
                    axes[0, 1].set_xlabel('Training Steps')
                    axes[0, 1].set_ylabel('mAP@0.5')
                    axes[0, 1].set_ylim([0, 1])
                    axes[0, 1].grid(True, alpha=0.3)

                # AP75 (mAP@0.75)
                if 'AP75' in df_metrics.columns and not df_metrics['AP75'].isna().all():
                    axes[1, 0].plot(df_metrics['step'], df_metrics['AP75'], linewidth=2,
                                   color='#EF476F', marker='o', markersize=4)
                    axes[1, 0].set_title('mAP@0.75 Over Training', fontsize=14, fontweight='bold')
                    axes[1, 0].set_xlabel('Training Steps')
                    axes[1, 0].set_ylabel('mAP@0.75')
                    axes[1, 0].set_ylim([0, 1])
                    axes[1, 0].grid(True, alpha=0.3)

                # All AP metrics together
                if all(col in df_metrics.columns for col in ['AP', 'AP50', 'AP75']):
                    axes[1, 1].plot(df_metrics['step'], df_metrics['AP'], linewidth=2,
                                   label='mAP@[0.5:0.95]', color='#06A77D', marker='o', markersize=3)
                    axes[1, 1].plot(df_metrics['step'], df_metrics['AP50'], linewidth=2,
                                   label='mAP@0.5', color='#118AB2', marker='s', markersize=3)
                    axes[1, 1].plot(df_metrics['step'], df_metrics['AP75'], linewidth=2,
                                   label='mAP@0.75', color='#EF476F', marker='^', markersize=3)
                    axes[1, 1].set_title('All mAP Metrics Comparison', fontsize=14, fontweight='bold')
                    axes[1, 1].set_xlabel('Training Steps')
                    axes[1, 1].set_ylabel('mAP')
                    axes[1, 1].set_ylim([0, 1])
                    axes[1, 1].legend()
                    axes[1, 1].grid(True, alpha=0.3)

                plt.tight_layout()
                plt.savefig(OUTPUT_DIR / 'training_map_metrics.png', dpi=300, bbox_inches='tight')
                plt.show()

            else:
                print("⚠ No metrics data found in event files")
        else:
            print("⚠ No metrics data found in event files")
    else:
        print("⚠ No TensorBoard event files found")

except Exception as e:
    print(f"⚠ Could not load TensorBoard logs: {e}")

## 11. Model Evaluation

### Evaluatio configuration

In [ ]:
# Set up evaluation paths and parameters
BEST_CHECKPOINT = CHECKPOINT_DIR / 'archive'
EVAL_MODEL_DIR = CHECKPOINT_DIR  # Use the trained model directory
EVAL_OUTPUT_DIR = OUTPUT_DIR / 'eval'
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

print("✅ Evaluation configuration:")
print(f"   Best checkpoint: {BEST_CHECKPOINT}")
print(f"   Model directory: {EVAL_MODEL_DIR}")
print(f"   Test TFRecord: {TEST_TFRECORD}")
print(f"   Test annotations: {TEST_ANNOTATIONS}")

### Evaluation on test set

In [ ]:
print("\n🔍 Running model evaluation on test set...")

!konda run "python automl/efficientdet/main.py \
    --mode=eval \
    --model_name={MODEL_VARIANT} \
    --val_file_pattern={TEST_TFRECORD} \
    --val_json_file={str(TEST_ANNOTATIONS)} \
    --model_dir={str(EVAL_MODEL_DIR)} \
    --eval_batch_size={BATCH_SIZE} \
    --hparams={HPARAMS_STR} \
    --eval_timeout=5"

print("✅ Evaluation completed!")

### Inference on images

In [ ]:
from PIL import Image

# Get random test image
test_image_path = random.choice(list(TEST_IMAGES_DIR.glob('*.jpg')))

test_image_filename = os.path.basename(test_image_path).split('.')[0]
resized_test_image_path = EVAL_OUTPUT_DIR / f"0-{test_image_filename}.jpg"

test_im = Image.open(test_image_path)
resized_im = test_im.resize((IMAGE_SIZE, IMAGE_SIZE))
resized_im.save(resized_test_image_path)

In [ ]:
INF_HPARAMS_DICT = dict (
    name = MODEL_VARIANT,
    num_classes = NUM_CLASSES,
    image_size = IMAGE_SIZE,
)

INF_HPARAMS_STR = ', '.join([f'{k}={v}' for k, v in HPARAMS_DICT.items()])
INF_HPARAMS_STR

In [ ]:
!konda run "MPLBACKEND=Agg python automl/efficientdet/model_inspect.py \
    --runmode=infer \
    --model_name={MODEL_VARIANT} \
    --hparams={INF_HPARAMS_STR} \
    --max_boxes_to_draw=100 \
    --min_score_thresh=0.4 \
    --ckpt_path={BEST_CHECKPOINT} \
    --input_image={resized_test_image_path} \
    --output_image_dir={EVAL_OUTPUT_DIR}"

In [ ]:
inferred_im = Image.open(EVAL_OUTPUT_DIR / "0.jpg")
plt.imshow(inferred_im)

## 12. Export SavedModel and TFLite

### Export configuration

In [ ]:
EXPORT_DIR = Path('/content/export')
os.makedirs(EXPORT_DIR, exist_ok=True)

SAVEDMODEL_DIR = EXPORT_DIR / 'saved_model'
os.makedirs(SAVEDMODEL_DIR, exist_ok=True)

TFLITE_PATH = EXPORT_DIR / f'deepoo_{MODEL_VARIANT}.tflite'

print("✅ Export configuration:")
print(f"   Export directory: {EXPORT_DIR}")
print(f"   SavedModel directory: {SAVEDMODEL_DIR}")
print(f"   TFlite path: {TFLITE_PATH}")

### Run export

In [ ]:
EXP_HPARAMS_DICT = dict (
    name = MODEL_VARIANT,
    num_classes = NUM_CLASSES,
    image_size = IMAGE_SIZE,
)

EXP_HPARAMS_STR = ', '.join([f'{k}={v}' for k, v in HPARAMS_DICT.items()])
EXP_HPARAMS_STR

In [ ]:
!konda run "MPLBACKEND=Agg python automl/efficientdet/model_inspect.py \
    --runmode=saved_model \
    --model_name={MODEL_VARIANT} \
    --ckpt_path={BEST_CHECKPOINT} \
    --hparams={EXP_HPARAMS_STR} \
    --saved_model_dir={SAVEDMODEL_DIR} \
    --tflite_path={TFLITE_PATH}"

### Inference with SavedModel

In [ ]:
!konda run "MPLBACKEND=Agg python automl/efficientdet/model_inspect.py \
    --runmode=saved_model_infer \
    --model_name={MODEL_VARIANT}  \
    --saved_model_dir={SAVEDMODEL_DIR} \
    --input_image={resized_test_image_path} \
    --output_image_dir={EVAL_OUTPUT_DIR}"


In [ ]:
savedmodel_inferred_im = Image.open(EVAL_OUTPUT_DIR / "0.jpg")
plt.imshow(savedmodel_inferred_im)

## 13. Evaluate TFLite Model

In [ ]:
EVAL_SCORE_THRESHOLD = 0.30
MAX_EVAL_IMAGES = None          # set to an int to cap images evaluated
SAVE_DETECTIONS_JSON = False    # set True to dump raw detections to disk
DETECTIONS_JSON_PATH = Path(OUTPUT_DIR) / "tflite_detections.json"
DEBUG_FIRST_IMAGE = False       # switch True if you need diagnostics again

In [ ]:
from tqdm import tqdm
from collections import Counter
import time as time_module


print("Loading TFLite model…")
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
input_index = input_details["index"]
input_height, input_width = input_details["shape"][1:3]
float_input = input_details["dtype"] in (np.float32, np.float64)
output_details = interpreter.get_output_details()

coco_gt = COCO(str(TEST_ANNOTATIONS))
filename_to_id = {img["file_name"]: int(img["id"]) for img in coco_gt.dataset["images"]}
category_map = {idx: int(cat_id) for idx, cat_id in enumerate(CATEGORY_IDS)}

feature_spec = {
    "image/encoded": tf.io.FixedLenFeature([], tf.string),
    "image/filename": tf.io.FixedLenFeature([], tf.string),
    "image/source_id": tf.io.FixedLenFeature([], tf.string),
    "image/height": tf.io.FixedLenFeature([], tf.int64),
    "image/width": tf.io.FixedLenFeature([], tf.int64),
}


def parse_example(example):
    parsed = tf.io.parse_single_example(example, feature_spec)
    image = tf.image.decode_image(parsed["image/encoded"], channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [input_height, input_width], method="bilinear")
    image = tf.cast(tf.clip_by_value(image, 0, 255), tf.uint8)
    return {
        "image": image,
        "filename": parsed["image/filename"],
        "source_id": parsed["image/source_id"],
        "orig_h": tf.cast(parsed["image/height"], tf.int32),
        "orig_w": tf.cast(parsed["image/width"], tf.int32),
    }


dataset = tf.data.TFRecordDataset(str(TEST_TFRECORD))
dataset = dataset.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
if MAX_EVAL_IMAGES is not None:
    dataset = dataset.take(MAX_EVAL_IMAGES)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

iterator = dataset.as_numpy_iterator()
if tqdm is not None:
    iterator = tqdm(iterator, desc="Evaluating TFLite")

detections, img_ids = [], []
total_time = 0.0
first_image_done = False
class_counter = Counter()

for image_count, example in enumerate(iterator):
    img = example["image"]
    orig_h, orig_w = int(example["orig_h"]), int(example["orig_w"])
    fname = example["filename"].decode("utf-8")
    source = example["source_id"].decode("utf-8")

    try:
        image_id = int(source)
    except ValueError:
        image_id = filename_to_id[fname]

    input_tensor = np.expand_dims(img, axis=0)
    if float_input:
        input_tensor = input_tensor.astype(np.float32) / 255.0
    else:
        input_tensor = input_tensor.astype(input_details["dtype"])

    interpreter.set_tensor(input_index, input_tensor)
    start = time_module.perf_counter()
    interpreter.invoke()
    total_time += time_module.perf_counter() - start

    det_tensor = interpreter.get_tensor(output_details[0]["index"])
    if DEBUG_FIRST_IMAGE and not first_image_done:
        print("Interpreter outputs:", output_details[0]["name"], det_tensor.shape, det_tensor.dtype)
        print("First image raw detections (top 5):")
        for det in det_tensor[0][:5]:
            print(tuple(float(x) for x in det))

    scale_x = orig_w / input_width
    scale_y = orig_h / input_height
    kept_for_image = 0

    for det in det_tensor[0]:
        _, ymin, xmin, ymax, xmax, score, class_idx = det
        ymin = float(ymin)
        xmin = float(xmin)
        ymax = float(ymax)
        xmax = float(xmax)
        score = float(score)
        class_idx = int(class_idx)

        if score < EVAL_SCORE_THRESHOLD:
            continue

        bbox = [
            xmin * scale_x,
            ymin * scale_y,
            (xmax - xmin) * scale_x,
            (ymax - ymin) * scale_y,
        ]
        category_id = category_map.get(class_idx, CATEGORY_IDS[0])

        if DEBUG_FIRST_IMAGE and not first_image_done and kept_for_image < 5:
            print("Scaled bbox:", bbox,
                  "score:", f"{score:.4f}",
                  "class_idx:", class_idx,
                  "category_id:", category_id)

        detections.append(
            {
                "image_id": image_id,
                "category_id": category_id,
                "bbox": bbox,
                "score": score,
            }
        )
        class_counter[category_id] += 1
        kept_for_image += 1

    if DEBUG_FIRST_IMAGE and not first_image_done:
        print(f"Detections kept for first image: {kept_for_image}")
        first_image_done = True

    img_ids.append(image_id)

if not detections:
    raise RuntimeError(
        "No detections passed the score threshold; lower EVAL_SCORE_THRESHOLD or verify the model."
    )

print(f"\nTotal detections kept: {len(detections)}")
print("Detections per category:", dict(class_counter))

if SAVE_DETECTIONS_JSON:
    DETECTIONS_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
    DETECTIONS_JSON_PATH.write_text(json.dumps(detections))
    print(f"Detections saved to {DETECTIONS_JSON_PATH}")

coco_dt = coco_gt.loadRes(detections)
evaluator = COCOeval(coco_gt, coco_dt, iouType="bbox")
evaluator.params.imgIds = img_ids
evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

avg_latency = total_time / len(img_ids)
print(f"\nImages evaluated: {len(img_ids)}")
print(f"Average latency: {avg_latency * 1000:.2f} ms")
print(f"Throughput: {1.0 / avg_latency:.2f} FPS")

### Visualize TFLite predictions on test images

In [ ]:
import random

NUM_SAMPLES = 6
PRED_SCORE_THRESHOLD = 0.30  # adjust as needed

sample_img_ids = random.sample(TEST_IMG_IDS, min(NUM_SAMPLES, len(TEST_IMG_IDS)))
cols = 3
rows = (len(sample_img_ids) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, 4.5 * rows))
axes = np.array(axes).reshape(rows, cols)

for ax in axes.flatten():
    ax.axis("off")

for idx, img_id in enumerate(sample_img_ids):
    ax = axes.flat[idx]
    img_info = COCO_TEST.loadImgs(img_id)[0]
    img_path = TEST_IMAGES_DIR / img_info["file_name"]
    image = np.array(Image.open(img_path).convert("RGB"))

    # --- run TFLite model ---
    resized = tf.image.resize(image, [input_height, input_width], method="bilinear").numpy()
    resized = np.clip(resized, 0, 255).astype(np.uint8)
    input_tensor = np.expand_dims(resized, axis=0)
    if float_input:
        input_tensor = input_tensor.astype(np.float32) / 255.0
    else:
        input_tensor = input_tensor.astype(input_details["dtype"])

    interpreter.set_tensor(input_index, input_tensor)
    interpreter.invoke()
    det_tensor = interpreter.get_tensor(output_details[0]["index"])[0]

    scale_x = image.shape[1] / input_width
    scale_y = image.shape[0] / input_height

    ax.imshow(image)
    ax.set_title(img_info["file_name"], fontsize=11)

    # --- draw ground-truth boxes (fuchsia) ---
    ann_ids = COCO_TEST.getAnnIds(imgIds=[img_id])
    anns = COCO_TEST.loadAnns(ann_ids)
    for ann in anns:
        x, y, w, h = ann["bbox"]
        rect = patches.Rectangle(
            (x, y),
            w,
            h,
            linewidth=2,
            edgecolor="fuchsia",
            facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(
            x,
            max(0, y - 6),
            COCO_TEST.loadCats(ann["category_id"])[0]["name"],
            color="white",
            fontsize=9,
            fontweight="bold",
            bbox=dict(facecolor="fuchsia", alpha=0.6, edgecolor="none", pad=2),
        )

    # --- draw TFLite predictions (blue) ---
    kept = 0
    for det in det_tensor:
        _, ymin, xmin, ymax, xmax, score, class_idx = det
        score = float(score)
        if score < PRED_SCORE_THRESHOLD:
            continue

        x = xmin * scale_x
        y = ymin * scale_y
        w = (xmax - xmin) * scale_x
        h = (ymax - ymin) * scale_y

        rect = patches.Rectangle(
            (x, y),
            w,
            h,
            linewidth=2,
            edgecolor="dodgerblue",
            facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(
            x,
            y + h + 6,
            f"TFLite {score:.2f}",
            color="white",
            fontsize=9,
            fontweight="bold",
            bbox=dict(facecolor="dodgerblue", alpha=0.6, edgecolor="none", pad=2),
        )
        kept += 1

    if kept == 0:
        ax.text(
            5,
            15,
            "No TFLite detections",
            color="dodgerblue",
            fontsize=10,
            fontweight="bold",
            bbox=dict(facecolor="dodgerblue", alpha=0.3, edgecolor="none", pad=2),
        )

plt.tight_layout()
plt.show()

## 14. Archive Model

In [ ]:
from datetime import datetime as dt

current_date = dt.today().strftime("%y%m%d")
ARCHIVE_NAME = f'DeePoo_{MODEL_VARIANT}_{current_date}'

### Create archive

In [ ]:
import shutil

# Copy train configuration to export directory
shutil.copy(CHECKPOINT_DIR / 'config.yaml', EXPORT_DIR)

# Copy the best checkpoint to export directory
BEST_CHECKPOINT_EXPORTS_DIR = EXPORT_DIR / 'archive'
shutil.copytree(BEST_CHECKPOINT, EXPORT_DIR, dirs_exist_ok=True)

# Create `zip` archive
shutil.make_archive(
    base_name=ARCHIVE_NAME,
    format='zip',
    root_dir=EXPORT_DIR
)

archive_path = f'{ARCHIVE_NAME}.zip'
print(f"Results archived to: {archive_path}")
print(f"Archive size: {os.path.getsize(archive_path) / (1024*1024):.2f} MB")

### Download archive

In [ ]:
from google.colab import files

files.download(archive_path)

print("\nDownload started! Check your browser downloads.")